# Water Quality Service 

In [ ]:
.. contents::
   :local:

**Date modified:** December 2025

## Service Overview

### Background

The Digital Earth Africa Water Quality Service provides information on the quality of water in areas identified as waterbodies in the DE Africa Waterbodies Service. It describes these waterbodies by a set of properties such as turbidity, presence of algal blooms, color, temperature of water, and levels of chlorophyll-a.

These properties are computed and mapped by applying published algorithms to multiple DE Africa Earth Observation (EO) data sources, making the service multi-sensor and sensor-agnostic. These properties are monitored to provide information on how they change over time and to generate alert notifications when they change beyond certain thresholds.

The water quality information is consistent with the reporting framework of the Sustainable Development Goals (SDGs). These measurements serve as indicators for environmental health and sustainable land and water management practices. The service provides ready-to-use data for decision makers, water managers, water users, environmental monitoring agencies, and the agricultural and pastoral sectors.

### Specifications

The Water Quality Service consists of multiple water quality parameters mapped annually for all waterbodies across Africa. A [Jupyter Notebook](https://docs.digitalearthafrica.org/en/latest/sandbox/notebooks/Datasets/WaterQuality.html) that demonstrates loading and using the Water Quality products in the Sandbox will be available.

**Table 1: Water Quality Service products**

|Product type |Description |Data type|
|:-------------|:-------------|:-------------|
|Turbidity |Total Suspended Solids (TSS) and turbidity measurements indicating water clarity |Raster (COG) |
|Chlorophyll-a / Trophic State |Chlorophyll-a concentration indicating algae levels and trophic state classification |Raster (COG)|
|Algal Blooms |Floating Algal Index (FAI) and NDVI indicating presence of vegetation/cyanobacteria |Raster (COG) |
|Water Color |Hue angle measurements characterizing water color |Raster (COG) |
|Optical Water Type |Classification of water optical properties |Raster (COG) |
|Water Temperature |Surface water temperature from thermal sensors |Raster (COG) |
|Water Mask |Analysis area |Raster (COG) |


**Table 2: Water Quality Service specifications**

||||
|-------------|-------------|-------------|
|**Service name** |DE Africa Water Quality Service |DE Africa Water Quality Service|
|**Product name** |wq_annual |annual water quality variables |
|**Coordinate reference system** |EPSG:6933 (Equal Area projection) | -- |
|**Spatial resolution** |10 m | Aggregated per waterbody |
|**Temporal resolution** |Annual | Annual |
|**Temporal range** |2000 - 2024 | 2000 - 2024 |
|**Parent datasets** |Landsat TM/ETM+/OLI, Sentinel-2 MSI Annual GeoMADs; Landsat Surface Temperature; WOfS Annual Summary |Same as raster products |
|**Update frequency** |Yearly | Yearly |

**Figure 1: DE Africa Water Quality Service geographic extent**

<img src="../_static/data_specs/Waterbodies_specs/waterbodies_extent.png" alt="Water Quality Service data extent." width="500" align="left"/>

The DE Africa Water Quality Service is a continental service that characterizes water quality parameters for individual water bodies across Africa. This service uses annual geomedian composites from multiple satellite sensors and has been generated for waterbodies identified in the [DE Africa Waterbodies Service](https://docs.digitalearthafrica.org/en/latest/data_specs/Waterbodies_specs.html). 

### Multi-Sensor Approach

The Water Quality Service adopts a sensor-agnostic approach that maximizes the use of available satellite data. Multiple algorithms are applied to data from different sensors from Landsat 5 TM (2000-2012), Landsat 7 ETM+ (2000-2024), Landsat 8-9 OLI (2013-2024), Sentinel-2 MSI (2017-2024) and WOfS annual to produce ensemble results, providing both central estimates and uncertainty ranges for each water quality parameter.


### Harmonization Process

To ensure consistency across different sensors and algorithms, a multi-level harmonization approach is employed:

**Level 0 - Inter-Sensor Harmonization:**
Adjusts outputs from the same algorithm applied to different sensors. All sensors are harmonized to match Sentinel-2 MSI as the reference sensor. Transformation functions are derived from overlapping observations in 2020 when data from TM, ETM+, OLI, and MSI were all available.

**Level 1 - Inter-Algorithm Harmonization:**
Adjusts probability distributions from different algorithms to a common form, allowing for inter-comparisons while maintaining ensemble variance.

**Level 2 - Reference Validation:**
Further adjustments based on empirical comparisons with Sentinel-3 OLCI data and in-situ measurements where available.

**Level 3 - Local Calibration:**
Fine-tuning for specific water bodies when local in-situ data are provided by users.

## Water Quality Algorithms

### Overview
The algorithms are applied to annual geomedian composites, which represent the typical or prevailing conditions for each year by reducing the influence of outliers such as clouds and extreme events.

### 1. Turbidity (Total Suspended Solids/Matter)

#### Description
Turbidity measures water clarity by quantifying the concentration of suspended particles (sediments, organic matter, and other particulates) in the water column. High turbidity reduces light penetration, affecting aquatic ecosystems and water quality. Turbidity is expressed in terms of Total Suspended Matter (TSM) or Total Suspended Solids (TSS), typically measured in mg/L.

#### Algorithms Implemented

**For Sentinel-2 MSI (2017-2024):**

1. **NDSSI_RG (Normalized Difference Suspended Sediment Index - Red-Green)**
   - Formula: `NDSSI_RG = (Red - Green) / (Red + Green)`
   - Bands: MSI Band 4 (665 nm) and Band 3 (560 nm)
   - Application: General turbidity estimation across various water types
   - Reference: Normalized difference approach for sediment detection

2. **SPM_Qiu Algorithm**
   - Formula: `SPM = 10^(2.26×(R/G)³ - 5.42×(R/G)² + 5.58×(R/G) - 0.72)`
   - Bands: MSI Band 4 (Red, 665 nm) and Band 3 (Green, 560 nm)
   - Application: Suspended particulate matter for coastal and inland waters
   - Note: Polynomial relationship in log space; discriminates well across turbidity ranges
   - Reference: Qiu, Z. et al. (2013)

3. **TSM_Lym Algorithm (Lymburner Total Suspended Matter)**
   - Formula: `TSM = 3957 × [(Green + Red)/2 × 0.0001]^1.6436`
   - Bands: MSI Band 3 (Green, 560 nm) and Band 4 (Red, 665 nm)
   - Application: Power law relationship, stable across observation ranges
   - Units: mg/L concentration
   - Reference: Lymburner et al. (2016) - Simple, stable models developed for Australian waters

4. **TSS_GR (Green-Red TSS)**
   - Formula: `TSS = (Green + Red) × (Red / Green)`
   - Bands: MSI Band 3 (Green) and Band 4 (Red)
   - Application: Band combination without exponential model fitting
   - Note: Based on Zhang et al. (2021) fundamentals but without the exponential transformation, which can produce unrealistic values

5. **TSS_GRB (Green-Red-Blue TSS)**
   - Formula: `TSS = (Green + Red) × (Red / Blue)`
   - Bands: MSI Band 2 (Blue), Band 3 (Green), and Band 4 (Red)
   - Application: Three-band combination for enhanced sensitivity
   - Note: Based on Zhang et al. (2023) but without exponential fitting, developed for Jiaozhou Bay

**For Landsat 8-9 OLI (2013-2024):**

1. **NDSSI_BNIR (Blue-SWIR variant)**
   - Formula: `NDSSI_BNIR = (SWIR - Blue) / (SWIR + Blue)`
   - Bands: OLI Band 6 (SWIR, 1610 nm) and Band 2 (Blue, 482 nm)
   - Application: Sediment detection exploiting SWIR response to turbidity

2. **NDSSI_RG (Red-Green variant)**
   - Formula: `NDSSI_RG = (Red - Green) / (Red + Green)`
   - Bands: OLI Band 4 (655 nm) and Band 3 (561 nm)

3. **TI_Yu (Turbidity Index)**
   - Formula: `TI = 0.01 × [(Red - Green) - (SWIR - Green)]^0.5`
   - Bands: OLI Band 6 (SWIR, 1610 nm), Band 4 (Red), Band 3 (Green)
   - Application: Developed for seamless retrieval across ocean to turbid river mouths
   - Reference: Yu, X. et al. (2019). An empirical algorithm to seamlessly retrieve the concentration of suspended particulate matter from water color across ocean to turbid river mouths. *Remote Sensing of Environment*, 235, 111491

4. **TSM_Lym Algorithm (OLI)**
   - Formula: `TSM = 3957 × [(Green + Red)/2 × 0.0001]^1.6436`
   - Bands: OLI Band 3 (Green) and Band 4 (Red)
   - Note: Same parameters as MSI; consistent across sensors

5. **SPM_Qiu Algorithm (OLI)**
   - Adapted for OLI band configuration
   - Bands: OLI Band 4 (Red) and Band 3 (Green)

6. **TSS_GR and TSS_GRB (OLI)**
   - Same formulations as MSI, adapted to OLI band wavelengths

**For Landsat 5 TM (2000-2012):**

1. **NDSSI_RG**
   - Formula: `NDSSI_RG = (Red - Green) / (Red + Green)`
   - Bands: TM Band 3 (660 nm) and Band 2 (560 nm)

2. **TI_Yu Algorithm**
   - Formula: `TI = 0.01 × [(Red - Green) - (NIR - Green)]^0.5`
   - Bands: TM Band 4 (NIR), Band 3 (Red), Band 2 (Green)

3. **TSM_Lym Algorithm (ETM/TM)**
   - Formula: `TSM = 3983 × [(Green + Red)/2 × 0.0001]^1.6246`
   - Bands: TM Band 2 (Green) and Band 3 (Red)
   - Note: Slightly different coefficients (3983 vs 3957; exponent 1.6246 vs 1.6436) optimized for ETM+ sensor characteristics

4. **SPM_Qiu Algorithm (TM)**
   - Adapted for TM band configuration

5. **TSS_GR and TSS_GRB (TM)**
   - Adapted for TM band configuration

#### Output Products
- **Consolidated Turbidity Layer:** Harmonized metric equivalent to SDG 6.6.1 turbidity indicator
- **Intermediate Layers:** Individual algorithm outputs from each sensor for ensemble analysis
- **Per-Waterbody Metrics:** Annual median turbidity values aggregated within each waterbody extent

#### Interpretation
- **Low turbidity (< 10 mg/L):** Clear water, typical of oligotrophic lakes
- **Moderate turbidity (10-50 mg/L):** Common in productive lakes and rivers
- **High turbidity (> 50 mg/L):** Indicates high sediment loads, often associated with erosion, runoff events, or algal blooms

#### Limitations
- Algorithms are calibrated for optically shallow to moderately turbid waters
- Extremely turbid waters (> 500 mg/L) may saturate sensor response
- Bottom reflectance in shallow, clear waters can affect measurements
- Dense vegetation in or around water can cause misclassification
- Zhang et al. models were developed for Jiaozhou Bay (China) with relatively low TSS values (<30 mg/L)

### 2. Chlorophyll-a (Trophic State)

#### Description
Chlorophyll-a concentration is a key indicator of phytoplankton biomass and is used to assess the trophic state of water bodies. High chlorophyll-a levels indicate eutrophic conditions, which can lead to algal blooms, oxygen depletion, and degraded water quality. Chlorophyll-a is typically measured in μg/L and is converted to a Trophic State Index for SDG reporting.

#### Algorithms Implemented

**For Sentinel-2 MSI (2017-2024):**

1. **NDCI (Normalized Difference Chlorophyll Index) - Multiple Red Edge Variants**
   
   MSI's red edge bands provide superior chlorophyll detection. Three variants are implemented:
   
   - **NDCI_54:** `NDCI = (Band 5 - Band 4) / (Band 5 + Band 4)`
     - Bands: MSI Band 5 (705 nm, Red Edge) and Band 4 (665 nm, Red)
     - Application: Sensitive to moderate chlorophyll concentrations
   
   - **NDCI_64:** `NDCI = (Band 6 - Band 4) / (Band 6 + Band 4)`
     - Bands: MSI Band 6 (740 nm, Red Edge) and Band 4 (665 nm, Red)
     - Application: Alternative red edge position for different water types
   
   - **NDCI_74:** `NDCI = (Band 7 - Band 4) / (Band 7 + Band 4)`
     - Bands: MSI Band 7 (783 nm, Red Edge) and Band 4 (665 nm, Red)
     - Application: Uses broader red edge band
   
   - Reference: Mishra, S., & Mishra, D. R. (2012). Normalized difference chlorophyll index: A novel model for remote estimation of chlorophyll-a concentration in turbid productive waters. *Remote Sensing of Environment*, 117, 394-406

2. **Chla_MERIS2B Algorithm (MERIS Two-Band Model)**
   - Formula: `Chl-a = 25.28 × (Band708/Band665)² + 14.85 × (Band708/Band665) - 15.18`
   - Bands: MSI Band 5 (705 nm, closest to MERIS 708 nm) and Band 4 (665 nm)
   - Application: Works well in Case 2 waters (coastal/inland)
   - Reference: Adapted from MERIS (Medium Resolution Imaging Spectrometer) Level 2 processing algorithms

3. **Chla_MODIS2B Algorithm (MODIS Two-Band Model)**
   - Formula: `Chl-a = 190.34 × (Band748/Band667) - 32.45`
   - Bands: MSI Band 6 (740 nm, closest to MODIS 748 nm) and Band 4 (665 nm)
   - Application: General chlorophyll estimation
   - Reference: MODIS ocean color processing adapted for MSI bands

4. **Chla_3BDA (Three Band Difference Algorithm)**
   - Formula: `Chl-a = (Blue × 0.0001) - (Red × 0.0001 × Green × 0.0001)`
   - Bands: MSI Band 2 (Blue, 490 nm), Band 3 (Green, 560 nm), Band 4 (Red, 665 nm)
   - Application: Effective for turbid productive waters
   - Reference: Byrne et al. (2024). LAQUA: a LAndsat water QUality retrieval tool for east African lakes

5. **Chla_Tebbs Algorithm (NIR/Red Ratio)**
   - Formula: `Chl-a = -135 + 451 × (NIR / Red)`
   - Bands: MSI Band 8A (864 nm, NIR) and Band 4 (665 nm, Red)
   - Application: Simple ratio-based approach
   - Reference: Tebbs et al. algorithms for East African lakes

6. **Chla_Toming Algorithm**
   - Formula: `Chl-a = Band5 - 0.5 × (Band4 / Band6)`
   - Bands: MSI Band 5 (705 nm), Band 4 (665 nm), Band 6 (740 nm)
   - Application: Red edge-based algorithm
   - Note: Appears to behave more like a TSS indicator in practice

**For Landsat 8-9 OLI (2013-2024):**

1. **NDCI_54 (NIR-Red Index)**
   - Formula: `NDCI = (NIR - Red) / (NIR + Red)`
   - Bands: OLI Band 5 (865 nm, NIR) and Band 4 (655 nm, Red)
   - Application: Approximates red edge response using NIR
   - Note: OLI lacks dedicated red edge bands; NIR is used as proxy, reducing sensitivity compared to MSI

2. **Chla_3BDA Algorithm**
   - Formula: Same as MSI
   - Bands: OLI Band 2 (Blue), Band 3 (Green), Band 4 (Red)

3. **Chla_Tebbs Algorithm**
   - Adapted for OLI band configuration
   - Bands: OLI Band 5 (NIR) and Band 4 (Red)

**For Landsat 5 TM (2000-2012):**

1. **NDCI_43 (NIR-Red Index)**
   - Formula: `NDCI = (NIR - Red) / (NIR + Red)`
   - Bands: TM Band 4 (830 nm, NIR) and Band 3 (660 nm, Red)
   - Application: NIR-Red ratio for chlorophyll estimation

2. **Chla_MODIS2B Algorithm (TM adaptation)**
   - Formula: `Chl-a = 190.34 × (Band4/Band3) - 32.45`
   - Bands: TM Band 4 (NIR, used as proxy for 748nm) and Band 3 (Red, 660 nm)
   - Note: Less accurate than MSI due to lack of red edge bands

3. **Chla_3BDA Algorithm**
   - Adapted for TM band configuration
   - Bands: TM Band 1 (Blue), Band 2 (Green), Band 3 (Red)

4. **Chla_Tebbs Algorithm**
   - Adapted for TM band configuration
   - Bands: TM Band 4 (NIR) and Band 3 (Red)

#### Trophic State Classification

Following SDG 6.6.1 methodology, chlorophyll-a concentrations are converted to a Trophic State Index (TSI) on a scale of 0-100:

| Chlorophyll-a (μg/L) | Trophic State Index | Classification | Description |
|---------------------|---------------------|----------------|-------------|
| < 0.04 | 0 | Ultra-oligotrophic | Extremely clear, very low productivity |
| 0.04 - 0.34 | 0-30 | Oligotrophic | Very clear, low productivity |
| 0.34 - 2.6 | 30-50 | Mesotrophic | Moderate clarity and productivity |
| 2.6 - 20 | 50-70 | Eutrophic | Reduced clarity, high productivity |
| 20 - 56 | 70-80 | Highly Eutrophic | Turbid, very high productivity |
| > 56 | 80-100 | Hypertrophic | Very turbid, excessive productivity |

The transformation follows a log-linear relationship:
```
TSI = 22.46 × log₁₀(Chl-a) + 30.91 + 0.53
```
where the 0.53 offset ensures lowest values align with the SDG reference data.

#### Output Products
- **Consolidated Trophic State Layer:** Harmonized metric equivalent to SDG 6.6.1 trophic state indicator
- **Intermediate Chlorophyll-a Layers:** Individual algorithm outputs from each sensor
- **Per-Waterbody Metrics:** Annual median, minimum, and maximum trophic state values

#### Interpretation
- **Oligotrophic (TSI < 40):** Nutrient-poor, clear water, low biological productivity
- **Mesotrophic (TSI 40-50):** Moderate nutrients, moderate productivity
- **Eutrophic (TSI 50-70):** Nutrient-rich, high productivity, potential for algal blooms
- **Hypertrophic (TSI > 70):** Very high nutrients, excessive algal growth, degraded water quality

#### Limitations
- Algorithms may be less accurate in waters with high suspended sediments (Case 2 waters)
- Red edge bands on MSI provide superior chlorophyll detection compared to OLI and TM
- Atmospheric correction errors can significantly affect blue-green band ratios
- Some algorithms may saturate at very high chlorophyll concentrations (> 100 μg/L)
- OLI and TM lack red edge bands, requiring NIR as a less-sensitive proxy

### 3. Algal Blooms and Vegetation

#### Description
Algal blooms, particularly cyanobacteria blooms, pose significant health and environmental risks. Dense surface accumulations of algae or floating vegetation can be detected using indices that exploit the characteristic spectral signatures in the near-infrared region. These indices help identify areas where algae or vegetation are present on or just below the water surface.

#### Algorithms Implemented

**1. NDVI (Normalized Difference Vegetation Index)**

- **Formula:** `NDVI = (NIR - Red) / (NIR + Red)`
- **MSI Bands:** Band 8 (842 nm) and Band 4 (665 nm)
- **OLI Bands:** Band 5 (865 nm) and Band 4 (655 nm)
- **TM Bands:** Band 4 (830 nm) and Band 3 (660 nm)
- **Application:** Detects vegetation and algae with chlorophyll
- **Interpretation:**
  - NDVI < 0: Typically water or non-vegetated surfaces
  - NDVI > 0: Indicates presence of vegetation or algae
  - NDVI > 0.3: Strong vegetation/algae signal

**2. FAI (Floating Algae Index)**

- **Formula:** `FAI = NIR - [Red + (SWIR - Red) × (λNIR - λRed) / (λSWIR - λRed)]`
- **MSI Bands:** 
  - NIR: Band 8 (842 nm)
  - Red: Band 4 (665 nm)
  - SWIR: Band 11 (1610 nm)
- **OLI Bands:**
  - NIR: Band 5 (865 nm)
  - Red: Band 4 (655 nm)
  - SWIR: Band 6 (1610 nm)
- **Application:** Specifically designed to detect floating algae or vegetation on water surface
- **Reference:** Hu et al. (2009) - developed for detecting floating macroalgae
- **Interpretation:**
  - FAI < 0: Clear water
  - FAI > 0: Floating algae or vegetation present
  - FAI > 0.01: Significant algal bloom or floating vegetation

#### Output Products
- **NDVI Layer:** Values above zero indicating vegetation/algae presence (2000-2024)
- **FAI Layer:** Values above zero indicating floating algae (2000-2024, limited by SWIR availability)
- **Per-Waterbody Metrics:** Percent of water area consistently indicating vegetation/algae during the year

#### Use Cases
- Early detection of cyanobacteria blooms
- Monitoring of aquatic vegetation encroachment
- Assessment of eutrophication impacts
- Water treatment plant intake management

#### Limitations
- FAI requires SWIR bands, which have coarser spatial resolution on some sensors
- Algorithms detect surface or near-surface phenomena; subsurface algae may not be detected
- Atmospheric effects (haze, thin clouds) can cause false positives
- Overhanging vegetation or shadows can be misclassified as algal blooms

### 3. Algal Blooms and Vegetation

#### Description
Algal blooms, particularly cyanobacteria blooms, pose significant health and environmental risks. Dense surface accumulations of algae or floating vegetation can be detected using indices that exploit the characteristic spectral signatures in the near-infrared region. These indices help identify areas where algae or vegetation are present on or just below the water surface.

#### Algorithms Implemented

**1. NDVI (Normalized Difference Vegetation Index)**

- **Formula:** `NDVI = (NIR - Red) / (NIR + Red)`
- **MSI Bands:** Band 8A (864 nm, NIR) and Band 4 (665 nm, Red)
- **OLI Bands:** Band 5 (865 nm, NIR) and Band 4 (655 nm, Red)
- **TM Bands:** Band 4 (830 nm, NIR) and Band 3 (660 nm, Red)
- **Application:** Detects vegetation and algae with chlorophyll
- **Implementation Notes:**
  - Values are normalized across sensors using reference means (MSI as baseline)
  - MSI reference mean: 0.2335, OLI: 0.2225, TM: 0.2000
  - Sensor-specific scaling applied: `scale = 0.2335 / reference_mean[sensor]`
  - Only values > 0.05 threshold retained
  - Weighted average computed based on observation counts when multiple sensors available
- **Interpretation:**
  - NDVI < 0: Typically water or non-vegetated surfaces
  - NDVI > 0: Indicates presence of vegetation or algae
  - NDVI > 0.3: Strong vegetation/algae signal

**2. FAI (Floating Algae Index)**

- **Formula:** `FAI = NIR - [Red + (SWIR - Red) × (λ_NIR - λ_Red) / (λ_SWIR - λ_Red)]`
  
  This is a baseline correction approach where the NIR reflectance is compared to a linear baseline between Red and SWIR.

- **MSI Implementation:**
  - NIR: Band 8A (λ = 864 nm)
  - Red: Band 4 (λ = 665 nm)
  - SWIR: Band 11 (λ = 1612 nm)
  - Baseline factors: (864-665)/(1612-665) = 0.210

- **OLI Implementation:**
  - NIR: Band 5 (λ = 865 nm, using band center)
  - Red: Band 4 (λ = 655 nm, using band center)
  - SWIR: Band 6 (λ = 1610 nm, using band center)
  - Baseline factors: (865-655)/(1610-655) = 0.220

- **TM Implementation:**
  - NIR: Band 4 (λ = 830 nm, using band center)
  - Red: Band 3 (λ = 660 nm, using band center)
  - SWIR: Band 5 (λ = 1650 nm, using band center)
  - Baseline factors: (830-660)/(1650-660) = 0.172

- **Application:** Specifically designed to detect floating algae or vegetation on water surface
- **Reference:** Hu, C. (2009). A novel ocean color index to detect floating algae in the global oceans. *Remote Sensing of Environment*, 113(10), 2118-2129
- **Implementation Notes:**
  - Values scaled by 10000 to produce typical range 0-1
  - Reference means for harmonization: MSI: 0.0970, OLI: 0.1015, TM: 0.0962
  - Scaling factor applied: `scale = 0.0970 / reference_mean[sensor]`
  - Only values > 0.05 threshold retained
  - Weighted average computed when multiple sensors available
- **Interpretation:**
  - FAI < 0: Clear water
  - FAI > 0: Floating algae or vegetation present
  - FAI > 0.01: Significant algal bloom or floating vegetation

#### Output Products
- **NDVI Layer:** Values above zero indicating vegetation/algae presence (2000-2024)
- **FAI Layer:** Values above zero indicating floating algae (2000-2024)
- **Sensor-Specific Layers:** Individual NDVI and FAI for MSI, OLI, and TM
- **Per-Waterbody Metrics:** Percent of water area consistently indicating vegetation/algae during the year

#### Use Cases
- Early detection of cyanobacteria blooms
- Monitoring of aquatic vegetation encroachment (e.g., water hyacinth)
- Assessment of eutrophication impacts
- Water treatment plant intake management
- Tourism and recreation advisories

#### Limitations
- FAI requires SWIR bands, which have coarser spatial resolution on some sensors
- Algorithms detect surface or near-surface phenomena; subsurface algae may not be detected
- Atmospheric effects (haze, thin clouds) can cause false positives
- Overhanging vegetation or shadows can be misclassified as algal blooms
- Harmonization across sensors uses empirical scaling factors which may vary regionally

### 4. Water Color (Hue Angle)

#### Description
Water color, expressed as hue angle, provides information about the optical properties and composition of water. Different constituents (sediments, algae, dissolved organic matter) produce characteristic colors that can be quantified using the hue angle in color space. Hue angle is measured in degrees (0-360°) and represents the dominant wavelength of light reflected by the water.

#### Algorithm Implementation

**Hue Angle Calculation:**

Water color is calculated by converting surface reflectance to CIE 1931 color space coordinates and then to hue angle:

1. **Convert to Tristimulus Values (X, Y, Z):**
   
   For each sensor, specific chromatic coefficients are applied to visible bands:
   ```
   X = Σ (R_band × coeff_X_band)
   Y = Σ (R_band × coeff_Y_band)
   Z = Σ (R_band × coeff_Z_band)
   ```

2. **Convert to Chromaticity Coordinates:**
   ```
   x = X / (X + Y + Z)
   y = Y / (X + Y + Z)
   ```

3. **Calculate Hue Angle:**
   ```
   Δx = x - x_white  (where x_white = 1/3)
   Δy = y - y_white  (where y_white = 1/3)
   hue = mod(arctan2(Δy, Δx) × 180/π + 360, 360)
   ```

4. **Apply Sensor-Specific Adjustment:**
   
   A quintic polynomial correction is applied to improve accuracy:
   ```
   Δhue = a₅×(hue/100)⁵ + a₄×(hue/100)⁴ + a₃×(hue/100)³ + a₂×(hue/100)² + a₁×(hue/100) + a₀
   hue_final = hue + Δhue
   ```

**Sensor-Specific Chromatic Coefficients:**

**MSI Coefficients (Bands 2,3,4,5):**
- Band 2 (490nm): X=12.040, Y=23.122, Z=61.055
- Band 3 (560nm): X=53.696, Y=65.702, Z=1.778
- Band 4 (665nm): X=32.028, Y=16.808, Z=0.015
- Band 5 (705nm): X=0.529, Y=0.192, Z=0.000
- Polynomial adjustment: [-161.23, 1117.08, -2950.14, 3612.17, -1943.57, 364.28]

**OLI Coefficients (Bands 1,2,3,4):**
- Band 1 (443nm): X=11.053, Y=1.320, Z=58.038
- Band 2 (482nm): X=6.950, Y=21.053, Z=34.931
- Band 3 (561nm): X=51.135, Y=66.023, Z=2.606
- Band 4 (655nm): X=34.457, Y=18.034, Z=0.016
- Polynomial adjustment: [-52.16, 373.81, -981.83, 1134.19, -533.61, 76.72]
- Note: OLI Band 1 (coastal/aerosol) not available in geomedian; hue calculation limited

**TM Coefficients (Bands 1,2,3):**
- Band 1 (485nm): X=13.104, Y=24.097, Z=63.845
- Band 2 (565nm): X=53.791, Y=65.801, Z=2.142
- Band 3 (660nm): X=31.304, Y=15.883, Z=0.013
- Polynomial adjustment: [-84.94, 594.17, -1559.86, 1852.50, -918.11, 151.49]

**Implementation Notes:**
- Hue values outside 25-100° range can produce unrealistic adjustments and are filtered
- When multiple sensors available, weighted average based on observation counts
- OLI geomedian lacks Band 1, reducing hue accuracy for 2013-2016 period

**Reference:** Van der Woerd, H. J., & Wernand, M. R. (2018). Hue-angle product for low to medium spatial resolution optical satellite sensors. *Remote Sensing*, 10(2), 180

#### Interpretation

| Hue Angle (degrees) | Dominant Color | Typical Water Composition |
|---------------------|----------------|---------------------------|
| 190-210 | Blue | Clear, oligotrophic water, low suspended matter |
| 210-230 | Blue-green | Moderate clarity, some phytoplankton |
| 230-260 | Green | High chlorophyll, algae-dominated |
| 260-290 | Yellow-green | Mixed suspended matter and algae |
| 290-330 | Yellow-brown | High sediment load, colored dissolved organic matter (CDOM) |
| 330-360 | Brown-red | Very high sediment, significant CDOM, or specialized algae |

#### Output Products
- **Hue Layer:** Hue angle measurements for MSI (2017-2024) and TM (2000-2012)
- **Sensor-Specific Hue:** Individual hue for MSI, OLI (limited), and TM
- **Per-Waterbody Metrics:** Annual median hue angle

#### Applications
- Water quality assessment and classification
- Tracking seasonal changes in water composition
- Identifying sources of pollution or runoff
- Complementary information to chlorophyll and turbidity measurements
- Tourism and recreation water quality monitoring

#### Limitations
- Gap in temporal coverage (2013-2016) due to OLI Band 1 unavailability in geomedian
- Atmospheric correction accuracy critically affects color measurements
- Bottom reflectance in shallow waters influences hue
- Sun glint and viewing geometry can affect measurements
- Polynomial adjustments can produce unrealistic values for extreme hue angles
- Requires careful quality control to filter invalid results

### 5. Optical Water Type (OWT)

#### Description
Optical Water Types classify water bodies based on their optical properties, which are determined by the relative contributions of chlorophyll, suspended sediments, and colored dissolved organic matter (CDOM). OWT classification helps in selecting appropriate algorithms for water quality parameter retrieval and provides a framework for understanding water body characteristics.

#### Algorithm Implementation

The OWT classification uses spectral shape analysis and clustering approaches to classify water based on its remote sensing reflectance spectrum. Multiple classification schemes are implemented:

**Classification Schemes:**

1. **OWT_MSI (2017-2024):**
   - Utilizes full MSI visible and red edge spectrum
   - Bands used: B2 (490nm), B3 (560nm), B4 (665nm), B5 (705nm), B6 (740nm), B8 (842nm)
   - Classification: Typically 7-13 water types based on spectral clustering
   - Approach: Fuzzy logic or hard classification using spectral angle or Euclidean distance

2. **OWT_OLI (2013-2024):**
   - Adapted for OLI band configuration
   - Bands used: B2 (482nm), B3 (561nm), B4 (655nm), B5 (865nm)
   - Classification: Modified scheme accounting for lack of red edge bands

3. **OWT_TM (2000-2012):**
   - Legacy classification for historical analysis
   - Bands used: B1 (485nm), B2 (560nm), B3 (660nm), B4 (830nm)
   - Classification: Simplified scheme due to broader bandwidths

#### Water Type Categories (Example Classification)

| OWT Class | Dominant Constituent | Typical Characteristics |
|-----------|---------------------|-------------------------|
| Type 1 | Clear water | Very low chlorophyll and TSM, high clarity |
| Type 2-3 | Low chlorophyll | Oligotrophic conditions, some CDOM |
| Type 4-5 | Moderate chlorophyll | Mesotrophic, balanced constituents |
| Type 6-8 | High chlorophyll | Eutrophic, algae-dominated |
| Type 9-10 | High sediments | Turbid, sediment-dominated |
| Type 11-12 | Mixed/CDOM | High organic content, stained water |
| Type 13 | Extreme conditions | Very high chlorophyll or sediments |

#### Classification Approach

1. **Spectral Normalization:**
   ```
   Rnorm(λ) = R(λ) / Σ[R(λ)]
   ```

2. **Distance Calculation:**
   - Spectral Angle Mapper (SAM)
   - Euclidean distance in n-dimensional space
   - Mahalanobis distance for probabilistic classification

3. **Membership Assignment:**
   - Hard classification: Single class assignment
   - Fuzzy classification: Probability of membership in each class

#### Output Products
- **OWT Layers:** Classification maps for each sensor/period
  - owt_msi (2017-2024)
  - owt_oli (2013-2024)
  - owt_tm (2000-2012)
- **Per-Waterbody Metrics:** Modal (most frequent) OWT class per year

#### Applications
- Algorithm selection for water quality parameter retrieval
- Water body classification and comparison
- Temporal monitoring of water optical property changes
- Identification of water bodies with similar characteristics for regional analysis

#### Limitations
- Classification schemes differ between sensors, making direct comparison challenging
- Atmospheric correction errors propagate into classification
- Mixed water types near boundaries may be difficult to classify
- Some water types may be rare in certain regions, limiting classification training

### 6. Water Surface Temperature

#### Description
Water surface temperature is a critical parameter affecting aquatic ecosystem health, stratification patterns, and biogeochemical processes. Temperature measurements from thermal infrared sensors provide information on diurnal and seasonal temperature variations, thermal pollution, and climate impacts on water bodies.

#### Algorithm Implementation

Water surface temperature is derived from Landsat thermal infrared sensors using split-window algorithms or single-channel methods:

**Data Sources:**

1. **Landsat 8-9 TIRS (2013-2024):**
   - Band 10 (10.9 μm) - primary thermal band
   - Band 11 (12.0 μm) - secondary thermal band (affected by stray light)
   - Spatial resolution: 100m (resampled to 30m)

2. **Landsat 7 ETM+ (2000-2024):**
   - Band 6 (10.4-12.5 μm) - single thermal band
   - Spatial resolution: 60m (resampled to 30m)

3. **Landsat 5 TM (2000-2011):**
   - Band 6 (10.4-12.5 μm) - single thermal band
   - Spatial resolution: 120m (resampled to 30m)

**Temperature Retrieval Methods:**

1. **Single-Channel Method:**
   ```
   Ts = γ × [ε⁻¹ × (ψ₁ × Lλ + ψ₂) + ψ₃] + δ
   ```
   where:
   - Ts = surface temperature (K)
   - ε = surface emissivity (typically 0.985-0.995 for water)
   - Lλ = at-sensor radiance
   - ψ₁, ψ₂, ψ₃ = atmospheric correction parameters
   - γ, δ = calibration constants

2. **Split-Window Method (TIRS only):**
   ```
   Ts = T₁₀ + c₁ × (T₁₀ - T₁₁) + c₂ × (T₁₀ - T₁₁)² + c₀ + (c₃ + c₄ × w) × (1 - ε) + (c₅ + c₆ × w) × Δε
   ```
   where:
   - T₁₀, T₁₁ = brightness temperatures from bands 10 and 11
   - c₀...c₆ = algorithm coefficients
   - w = atmospheric water vapor content
   - ε = average emissivity
   - Δε = emissivity difference between bands

**Emissivity Assumptions:**
- Clear water: ε = 0.991-0.995
- Turbid water: ε = 0.985-0.990
- Water with surface films: ε = 0.980-0.985

#### Output Products
- **Annual Minimum Temperature:** Lowest median temperature during the year
- **Annual Median Temperature:** Typical temperature for the year
- **Annual Maximum Temperature:** Highest median temperature during the year
- **Per-Waterbody Metrics:** Min, median, and max temperatures aggregated per waterbody

#### Interpretation

**Seasonal Patterns:**
- Tropical regions: 25-32°C with minimal seasonal variation
- Subtropical regions: 15-30°C with moderate seasonal variation
- Temperate regions: 0-28°C with strong seasonal variation

**Ecological Thresholds:**
- < 10°C: Cold water conditions, limited biological activity
- 10-20°C: Optimal for temperate species
- 20-30°C: Warm water, high productivity, potential for algal blooms
- > 30°C: Thermal stress for many aquatic organisms

#### Applications
- Monitoring thermal pollution from power plants or industrial discharge
- Understanding seasonal stratification patterns
- Assessing climate change impacts on aquatic ecosystems
- Predicting harmful algal bloom conditions
- Habitat suitability assessment for aquatic species

#### Limitations
- Thermal measurements represent skin temperature (top few micrometers)
- Single satellite overpass provides snapshot, not diurnal range
- Landsat overpass time (10:00-11:00 local time) may not capture maximum daily temperature
- Clouds completely block thermal measurements
- Coarser spatial resolution compared to optical bands
- Atmospheric water vapor significantly affects accuracy
- TIRS Band 11 stray light issues limit split-window algorithm effectiveness

## Data Access and Formats

### Raster Layers

**Storage Format:**
- Stored in tiled Cloud Optimized GeoTIFF (COG) format in AWS S3
- Indexed in Open Data Cube (ODC) for efficient access
- Available through Open Web Services (OWS) for visualization
- Coordinate Reference System: EPSG:6933 (Africa Albers Equal Area Conic)
- Spatial Resolution: 10m

**Available Layers:**
- Water mask (defines analysis area)
- NDVI (vegetation/algae indicator)
- FAI (floating algal index)
- Hue (water color)
- Optical Water Type (OWT classifications)
- Trophic state (consolidated metric)
- Turbidity (consolidated metric)
- Water temperature (min, median, max)
- Intermediate algorithm-specific layers for ensemble analysis

**STAC Metadata:**
All products include SpatioTemporal Asset Catalog (STAC) metadata for discovery and access.

### Per-Waterbody Summary Metrics

**Storage Format:**
- Stored in database table, linked to DE Africa Waterbodies table via geohash
- Accessible through API (similar to waterbody extent query)
- Available in DE Africa Map interface as waterbody attributes

**Available Metrics:**
- Percent of water area with high NDVI/FAI (vegetation/algae)
- Annual median hue
- Annual median Optical Water Type
- Annual median, minimum, and maximum trophic state
- Annual median, minimum, and maximum turbidity
- Annual minimum, median, and maximum water temperature



### Amazon Web Service S3
The Digital Earth Africa Waterbodies products can be accessed from the associated S3 bucket.

**Table 5: AWS data access details**

|AWS S3 details | |
|----------|-------------|
|Bucket ARN | `arn:aws:s3:::deafrica-services`|
| Product names| `waterbodies` |
| Region| `af-south-1` |

### OGC Web Services (OWS)

This product is available through DE Africa's OWS.

**Table 6: OWS data access details**

|OWS details | |
|----------|-------------|
|Name | `DE Africa Services` |
|Web Map Services (WMS) URL | `https://geoserver.digitalearth.africa/geoserver/wms` |
| Web Feature Services (WFS) URL | `https://geoserver.digitalearth.africa/geoserver/wfs`|
| Layer names | `DEAfrica_Waterbodies` |

### DE Africa Sandbox

The Waterbodies Service can be loaded and analysed in the DE Africa Sandbox following the [example Jupyter Notebook](https://github.com/digitalearthafrica/deafrica-sandbox-notebooks).
 
For further information regarding the use of DE Africa Water Bodies Service, [visit the Digital Earth Africa Help Desk](https://helpdesk.digitalearthafrica.org/portal/en/home).

## References


### Related DE Africa Products

- **Waterbodies Service:** https://docs.digitalearthafrica.org/en/latest/data_specs/Waterbodies_specs.html
- **Water Observations from Space (WOfS):** https://docs.digitalearthafrica.org/en/latest/data_specs/Landsat_WOfS_specs.html


### Contact and Support

For questions, feedback, or support:
